In [1]:
import mlflow
# Step 1: Set up the MLflow tracking server
mlflow.set_tracking_uri("http://184.72.71.39:5000/")

c:\Users\Abhi\Documents\youtube-comment-analysis\mlops-youtube-comment-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set or create an experiment
mlflow.set_experiment("ML Algos with HP Tuning")

<Experiment: artifact_location='s3://comment-analysis-bucket-994/6', creation_time=1788963985769, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1788963985769, lifecycle_stage='active', name='ML Algos with HP Tuning', tags={}, trace_location=None, workspace='default'>

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import mlflow
import mlflow.sklearn
import optuna


In [4]:
df = pd.read_csv('../data/processed/processed_comments.csv').dropna()
df.shape

(36662, 2)

In [5]:
# Remove rows with missing values
df = df.dropna(subset=['category', 'clean_comment'])

ngram_range = (1, 3)
max_features = 1000

# Final train-test split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_comment'],
    df['category'],
    test_size=0.2,
    random_state=42,
    stratify=df['category']
)

# Validation split for Optuna
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# TF-IDF on inner training data only
vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_inner_vec = vectorizer.fit_transform(X_train_inner)
X_val_vec = vectorizer.transform(X_val)

# SMOTE only on inner training data
smote = SMOTE(random_state=42)

X_train_inner_vec, y_train_inner = smote.fit_resample(
    X_train_inner_vec,
    y_train_inner
)


# Optuna objective for KNN
def objective_knn(trial):

    n_neighbors = trial.suggest_int(
        'n_neighbors',
        3,
        30
    )

    p = trial.suggest_categorical(
        'p',
        [1, 2]
    )

    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        p=p
    )

    model.fit(X_train_inner_vec, y_train_inner)

    y_pred = model.predict(X_val_vec)

    return accuracy_score(y_val, y_pred)


# Run Optuna
study = optuna.create_study(direction="maximize")

study.optimize(
    objective_knn,
    n_trials=30
)

best_params = study.best_params

print("Best parameters:", best_params)
print("Best validation accuracy:", study.best_value)


# Train final model using all training data
final_vectorizer = TfidfVectorizer(
    ngram_range=ngram_range,
    max_features=max_features
)

X_train_vec = final_vectorizer.fit_transform(X_train)
X_test_vec = final_vectorizer.transform(X_test)

smote = SMOTE(random_state=42)

X_train_vec, y_train = smote.fit_resample(
    X_train_vec,
    y_train
)

best_model = KNeighborsClassifier(
    n_neighbors=best_params['n_neighbors'],
    p=best_params['p']
)

best_model.fit(X_train_vec, y_train)

y_pred = best_model.predict(X_test_vec)

accuracy = accuracy_score(y_test, y_pred)

print("Final accuracy:", accuracy)


# Log results in MLflow
with mlflow.start_run():

    mlflow.set_tag(
        "mlflow.runName",
        "KNN_SMOTE_TFIDF_Trigrams"
    )

    mlflow.set_tag(
        "experiment_type",
        "algorithm_comparison"
    )

    mlflow.log_param("algo_name", "KNN")
    mlflow.log_param("ngram_range", str(ngram_range))
    mlflow.log_param("max_features", max_features)
    mlflow.log_param("n_trials", 30)

    mlflow.log_params(best_params)

    mlflow.log_metric("accuracy", accuracy)

    classification_rep = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )

    for label, metrics in classification_rep.items():
        if isinstance(metrics, dict):
            for metric, value in metrics.items():
                mlflow.log_metric(
                    f"{label}_{metric}",
                    value
                )

    mlflow.sklearn.log_model(
        best_model,
        name="KNN_model",
        skops_trusted_types=[
            "scipy.sparse._csr.csr_matrix"
        ]
    )

[I 2026-09-09 18:28:02,205] A new study created in memory with name: no-name-29a27257-04ef-4475-914e-ec30eeb0ab5c
[I 2026-09-09 18:28:04,344] Trial 0 finished with value: 0.40862598022502555 and parameters: {'n_neighbors': 13, 'p': 1}. Best is trial 0 with value: 0.40862598022502555.
[I 2026-09-09 18:28:07,418] Trial 1 finished with value: 0.4659052165018752 and parameters: {'n_neighbors': 10, 'p': 2}. Best is trial 1 with value: 0.4659052165018752.
[I 2026-09-09 18:28:09,675] Trial 2 finished with value: 0.465564268666894 and parameters: {'n_neighbors': 8, 'p': 2}. Best is trial 1 with value: 0.4659052165018752.
[I 2026-09-09 18:28:11,579] Trial 3 finished with value: 0.37828162291169454 and parameters: {'n_neighbors': 29, 'p': 1}. Best is trial 1 with value: 0.4659052165018752.
[I 2026-09-09 18:28:14,100] Trial 4 finished with value: 0.46130242072962835 and parameters: {'n_neighbors': 12, 'p': 2}. Best is trial 1 with value: 0.4659052165018752.
[I 2026-09-09 18:28:15,999] Trial 5 fin

Best parameters: {'n_neighbors': 4, 'p': 2}
Best validation accuracy: 0.4717013296965564
Final accuracy: 0.4732033274239738
🏃 View run KNN_SMOTE_TFIDF_Trigrams at: http://184.72.71.39:5000/#/experiments/6/runs/db4a072b51e243a0b951168e3d7491f8
🧪 View experiment at: http://184.72.71.39:5000/#/experiments/6
